# TFM — Fase 1: Exploración del Dataset

**Generación de Informes Radiológicos mediante Arquitecturas Encoder-Decoder Multimodales**

Daniel Gómez Arranz · Máster Universitario en Inteligencia Artificial

---
Este notebook realiza el análisis exploratorio (EDA) del dataset **Open-I** (Indiana University Chest X-Ray Collection).  
Cubre:
1. Carga y estadísticas del dataset
2. Análisis del texto de los informes
3. Exploración visual de las imágenes
4. Distribución de secciones (Findings / Impression)
5. Verificación del pipeline de preprocesamiento

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from collections import Counter

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA_ROOT = Path('../data/raw/open-i')
print(f'Dataset root: {DATA_ROOT.resolve()}')
print(f'Exists: {DATA_ROOT.exists()}')

## 1. Carga de Metadatos

In [ ]:
from src.data.open_i import build_open_i_metadata

df = build_open_i_metadata(DATA_ROOT)
print(f'Total image-report pairs: {len(df)}')
print(f'Unique studies (UIDs): {df["uid"].nunique()}')
df.head()

In [ ]:
# Basic statistics
print('=== Dataset Statistics ===')
print(f'Total samples:          {len(df):>6}')
print(f'Unique studies (UIDs):  {df["uid"].nunique():>6}')
print(f'Imgs with findings:     {(df["findings"].str.len() > 0).sum():>6}')
print(f'Imgs with impression:   {(df["impression"].str.len() > 0).sum():>6}')
print(f'Imgs per study (mean):  {len(df)/df["uid"].nunique():>6.2f}')

views_per_study = df.groupby('uid').size()
print(f'Studies with 1 image:   {(views_per_study == 1).sum():>6}')
print(f'Studies with 2 images:  {(views_per_study == 2).sum():>6}')
print(f'Studies with >2 images: {(views_per_study > 2).sum():>6}')

## 2. Análisis de Texto — Longitud de Informes

In [ ]:
# Per-study text (avoid counting duplicated text from multi-view studies)
df_study = df.drop_duplicates('uid').copy()

df_study['n_words_findings'] = df_study['findings'].str.split().str.len().fillna(0)
df_study['n_words_impression'] = df_study['impression'].str.split().str.len().fillna(0)
df_study['n_words_report'] = df_study['report'].str.split().str.len().fillna(0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(
    axes,
    ['n_words_findings', 'n_words_impression', 'n_words_report'],
    ['Findings', 'Impression', 'Full Report']
):
    data = df_study[col].dropna()
    ax.hist(data, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(data.median(), color='tomato', linestyle='--', label=f'Median: {data.median():.0f}')
    ax.set_title(f'{title} — Word Count')
    ax.set_xlabel('Words')
    ax.set_ylabel('Frequency')
    ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/eda_report_lengths.png', bbox_inches='tight')
plt.show()

df_study[['n_words_findings', 'n_words_impression', 'n_words_report']].describe().round(1)

## 3. Palabras más frecuentes en Informes

In [ ]:
import re
from src.data.preprocessing import ReportPreprocessor

proc = ReportPreprocessor()
all_words = []
for text in df_study['report'].dropna():
    cleaned = proc.clean(text)
    all_words.extend(re.findall(r'\b[a-zA-Z]{3,}\b', cleaned.lower()))

# Remove generic stopwords
STOPWORDS = {
    'the', 'and', 'are', 'was', 'with', 'this', 'that', 'for', 'not',
    'there', 'has', 'have', 'been', 'within', 'from', 'without', 'their',
    'which', 'also', 'any', 'both', 'than', 'can', 'but', 'its'
}
word_freq = Counter(w for w in all_words if w not in STOPWORDS)
top_words = pd.DataFrame(word_freq.most_common(30), columns=['word', 'count'])

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=top_words, x='count', y='word', palette='Blues_r', ax=ax)
ax.set_title('Top 30 Clinical Terms in Open-I Reports')
ax.set_xlabel('Frequency')
plt.tight_layout()
plt.savefig('../data/processed/eda_top_words.png', bbox_inches='tight')
plt.show()

## 4. Exploración Visual de Imágenes

In [ ]:
# Sample random images and display with their impression
sample = df[df['impression'].str.len() > 20].sample(6, random_state=42)

fig = plt.figure(figsize=(18, 10))
for i, (_, row) in enumerate(sample.iterrows()):
    ax = fig.add_subplot(2, 3, i + 1)
    img = cv2.imread(row['image_path'], cv2.IMREAD_GRAYSCALE)
    if img is not None:
        ax.imshow(img, cmap='gray')
        impression_short = (row['impression'][:120] + '...') if len(row['impression']) > 120 else row['impression']
        ax.set_title(f'UID: {row["uid"]}\n{impression_short}', fontsize=7, wrap=True)
    ax.axis('off')
plt.suptitle('Open-I Sample Images with Impressions', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/eda_sample_images.png', bbox_inches='tight')
plt.show()

In [ ]:
# Image size distribution
heights, widths = [], []
for path in df['image_path'].sample(min(200, len(df)), random_state=0):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        heights.append(img.shape[0])
        widths.append(img.shape[1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(heights, bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Image Height Distribution')
axes[0].set_xlabel('Pixels')
axes[1].hist(widths, bins=20, color='seagreen', edgecolor='white')
axes[1].set_title('Image Width Distribution')
axes[1].set_xlabel('Pixels')
for ax in axes:
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

print(f'Height — mean: {np.mean(heights):.0f}px, std: {np.std(heights):.0f}px')
print(f'Width  — mean: {np.mean(widths):.0f}px, std: {np.std(widths):.0f}px')

## 5. Verificación del Pipeline de Preprocesamiento

In [ ]:
from src.data.preprocessing import CXRPreprocessor

sample_row = df[df['image_exists']].iloc[0]
raw_img = CXRPreprocessor.load_image(sample_row['image_path'])

prep_train = CXRPreprocessor(image_size=224, mode='train')
prep_eval  = CXRPreprocessor(image_size=224, mode='eval')

tensor_train = prep_train(raw_img)
tensor_eval  = prep_eval(raw_img)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(raw_img, cmap='gray')
axes[0].set_title(f'Original — {raw_img.shape}')
axes[1].imshow(tensor_eval.permute(1,2,0).numpy() * 0.5 + 0.5)
axes[1].set_title(f'Eval transform — {tuple(tensor_eval.shape)}')
axes[2].imshow(tensor_train.permute(1,2,0).numpy() * 0.5 + 0.5)
axes[2].set_title(f'Train transform (augmented) — {tuple(tensor_train.shape)}')
for ax in axes: ax.axis('off')
plt.suptitle('Preprocessing Pipeline Verification', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Tensor dtype: {tensor_eval.dtype}, shape: {tensor_eval.shape}')
print(f'Value range: [{tensor_eval.min():.3f}, {tensor_eval.max():.3f}]')

In [ ]:
# Verify report preprocessing
from src.data.preprocessing import ReportPreprocessor

proc = ReportPreprocessor(extract_sections=True)
sample_report = df_study.iloc[10]['report']

print('=== RAW REPORT ===')
print(sample_report[:400])
print()

sections = proc.process(sample_report)
print('=== CLEANED FULL ===')
print(sections['full'][:300])
print()
print('=== FINDINGS ===')
print(sections['findings'][:200])
print()
print('=== IMPRESSION ===')
print(sections['impression'][:200])

## 6. Guardado del Split de Datos

In [ ]:
from src.data.open_i import OpenIDataModule
from transformers import AutoTokenizer

# Use GPT-2 tokenizer as proxy (will be replaced by final decoder tokenizer)
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

dm = OpenIDataModule(
    data_root=str(DATA_ROOT),
    tokenizer=tokenizer,
    batch_size=16,
    num_workers=0,
    splits_cache='../data/splits/open_i_splits.json',
)
dm.prepare_data()
dm.setup()

print('Split sizes:', dm.split_sizes)

# Verify a batch loads correctly
batch = next(iter(dm.train_dataloader()))
print(f'Image batch:      {batch["image"].shape}')
print(f'Input IDs batch:  {batch["input_ids"].shape}')
print(f'Attention masks:  {batch["attention_mask"].shape}')
print(f'Sample report:    {batch["report_text"][0][:100]}...')